# RAG Walkthrough

End-to-end RAG implementation from scratch.

**Steps:**
1. Load documents
2. Chunk documents
3. Create embeddings
4. Store in vector DB
5. Query → Retrieve → Generate

In [ ]:
from openai import OpenAI
import chromadb
import os

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
chroma = chromadb.Client()
collection = chroma.create_collection('docs')

print('Setup complete')

## 1. Load and Chunk Documents

In [ ]:
documents = [
    'Azure AI Foundry is a unified platform for building, evaluating, and deploying AI solutions at scale.',
    'It provides tools for prompt engineering, RAG patterns, fine-tuning, and model monitoring.',
    'Azure AI Search enables hybrid vector and keyword search for enterprise RAG applications.',
    'Content Safety filters detect and block harmful content in AI-generated outputs.',
]

chunks = []
for i, doc in enumerate(documents):
    chunks.append({'id': str(i), 'text': doc})

print(f'Created {len(chunks)} chunks')

## 2. Create Embeddings and Index

In [ ]:
def embed(text):
    resp = client.embeddings.create(
        model='text-embedding-3-small',
        input=text
    )
    return resp.data[0].embedding

for chunk in chunks:
    emb = embed(chunk['text'])
    collection.add(
        ids=[chunk['id']],
        embeddings=[emb],
        documents=[chunk['text']]
    )

print('Indexed all chunks')

## 3. Query and Generate

In [ ]:
query = 'How does Azure support RAG?'
query_emb = embed(query)
results = collection.query(query_embeddings=[query_emb], n_results=2)

context = '\n\n'.join(results['documents'][0])
print(f'Retrieved context:\n{context}\n')

In [ ]:
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': 'Answer based on the provided context.'},
        {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'}
    ],
    temperature=0,
)

print(f'Answer: {response.choices[0].message.content}')